# RoboMaster 行为树自定义插件文档

本文档基于 `3v3_new.xml` 行为树配置文件，对所有自定义节点进行全面解析。

**文档包含：**
1. XML 解析 → 提取全部自定义节点定义
2. 按类型分类展示（Action / Condition / Decorator）
3. 对应 `.hpp` / `.cpp` 源文件扫描
4. 各节点功能、端口、用法详细说明
5. 子树 XML 文件检查
6. 黑板端口一致性检测
7. **发现的逻辑错误与潜在问题汇总**
8. 节点速查表

---

## 📋 节点目录索引

> **更新说明**: 已移除 12 个无源文件的节点声明（Rotate、ScanStatus、SetGoal、SubHP、SelectBattleMode、RefSerialBlackboardSync、StopMotion、UpdateWasDead、RecoveryTimeoutGuard、IsGameStart、IsModeRight、IsModeLeft），已修复全部可修复的 Bug。

### Action 节点 (19个)

**数据订阅类**
- `SubGameStatus` - 订阅比赛状态（阶段、剩余时间） ✅
- `SubRobotStatus` - 订阅机器人状态（血量、热量、攻击状态） ✅
- `SubArmors` - 订阅装甲板检测结果 ✅
- `SubAllRobotHP` - 订阅全体机器人血量 ✅
- `SubDecisionNum` - 订阅决策编号（模式切换） ✅
- `SubRFIDStatus` - 订阅RFID状态（补给区刷卡） ✅
- `SubRobotPosition` - 订阅机器人位置（x, y, yaw） ✅ 已修复线程安全

**控制输出类**
- `SendGoal` - 发布导航目标点（PoseStamped） ✅ 已修复frame_id
- `RobotControl` - 发布云台/底盘控制指令 ✅
- `NavControlCmd` - 发布导航控制命令 ✅
- `CancelNavGoal` - 取消Nav2导航目标 ✅

**位置与移动**
- `GetCurrentLocation` - 通过TF2获取当前位姿 ⚠️ TF spin问题(设计层面)
- `MoveAround` - 随机小范围移动 ✅ 已修复阻塞问题

**配置与初始化**
- `InitBlackboardConfig` - 一次性初始化所有配置参数 ✅
- `SetNavGoalFromConfig` - 从配置读取并设置导航目标（含安全限幅） ✅
- `InitSearchTimerIfNeeded` - 初始化搜卡计时器 ✅

**恢复与补给逻辑**
- `DetectRespawnAndSetRecovery` - 检测复活并设置恢复标志（防抖） ✅
- `ClearRecoveryFlag` - 清除恢复标志和计时器 ✅
- `MicroSearchSupplyCard` - 补给区十字微移搜索RFID卡 ✅
- `WaitAndHeal` - 补给区等待回血（双门限判定） ✅

**特殊控制**
- `KeepRunning` - 永远返回RUNNING（阻塞分支用） ✅ 已修复基类错误

### Condition 节点 (11个)

**血量判断**
- `IsHPBelow` - 判断血量 < 阈值 ✅
- `IsHPAbove` - 判断血量 >= 阈值 ✅
- `IsDead` - 判断是否死亡（HP ≤ 0） ✅

**游戏状态**
- `IsGameTime` - 判断比赛阶段和剩余时间区间 ✅

**敌情侦测**
- `IsDetectEnemy` - 判断是否检测到敌人（含时间戳验证） ⚠️ 类型需确认
- `IsAttacked` - 判断是否被攻击 ✅ 已修复拼写错误

**综合状态**
- `IsStatusOK` - 综合判断血量/前哨站/热量 ✅ 已修复XML端口
- `IsFriendOK` - 比较双方平均血量判断血量优势 ✅

**位置与导航**
- `IsWithinScope` - 判断是否在目标点有效半径内 ⚠️ 坐标需配置
- `NotArrived` - 判断是否未到达补给区 ✅ 已修复端口
- `IsSupplyCardDetected` - 判断RFID补给区卡是否刷到 ✅

**恢复**
- `IsRecoveryNeeded` - 读取恢复标志（need_recovery） ✅

### Decorator 节点 (1个)

- `RateController` - 按指定频率（hz）tick子节点 ✅ 已修复动态频率

### Control 节点 (1个)

- `DecisionSwitch` - 按决策编号切换子节点 ✅ 已修复越界检查

---

**图例说明：**
- ✅ = 正常 / 已修复
- ⚠️ = 设计层面问题（不影响基本功能）

**共计：** 32个自定义节点（Action: 19, Condition: 11, Decorator: 1, Control: 1）
*已从XML中移除12个无源文件的声明节点*

---

## 1. 解析 XML 提取所有自定义节点定义

使用 Python `xml.etree.ElementTree` 解析 `3v3_new.xml`，从 `<TreeNodesModel>` 中提取所有自定义节点。

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path
import re

# 解析 XML
xml_path = Path(r"d:\open_resource_project\ros2_ws\src\rm_behavior_tree\rm_behavior_tree\config\3v3_new.xml")
tree = ET.parse(xml_path)
root = tree.getroot()

# 提取 TreeNodesModel 中的所有节点
nodes_model = root.find("TreeNodesModel")
all_nodes = {}

for node_type in ["Action", "Condition", "Decorator", "Control"]:
    for node in nodes_model.findall(node_type):
        node_id = node.get("ID")
        ports = {
            "input": [],
            "output": [],
            "inout": []
        }
        for port in node.findall("input_port"):
            ports["input"].append({
                "name": port.get("name"),
                "default": port.get("default", "")
            })
        for port in node.findall("output_port"):
            ports["output"].append({
                "name": port.get("name"),
                "default": port.get("default", "")
            })
        for port in node.findall("inout_port"):
            ports["inout"].append({
                "name": port.get("name"),
                "default": port.get("default", "")
            })
        
        all_nodes[node_id] = {
            "type": node_type,
            "ports": ports,
            "total_ports": len(ports["input"]) + len(ports["output"]) + len(ports["inout"])
        }

print(f"共提取 {len(all_nodes)} 个自定义节点：")
for nid, info in all_nodes.items():
    port_names = []
    for kind in ["input", "output", "inout"]:
        for p in info["ports"][kind]:
            port_names.append(f"{kind}:{p['name']}")
    print(f"  [{info['type']:10s}] {nid:35s} | 端口数={info['total_ports']} | {', '.join(port_names)}")

## 2. 按节点类型分类（Action / Condition / Decorator）

In [ ]:
import pandas as pd

rows = []
for nid, info in all_nodes.items():
    port_names = []
    for kind in ["input", "output", "inout"]:
        for p in info["ports"][kind]:
            port_names.append(f"{p['name']}({kind})")
    rows.append({
        "节点ID": nid,
        "类型": info["type"],
        "端口数": info["total_ports"],
        "端口列表": ", ".join(port_names)
    })

df = pd.DataFrame(rows)
df = df.sort_values(by=["类型", "节点ID"]).reset_index(drop=True)
print(f"\n=== Action 节点 ({len(df[df['类型']=='Action'])}) ===")
display(df[df['类型']=='Action'])
print(f"\n=== Condition 节点 ({len(df[df['类型']=='Condition'])}) ===")
display(df[df['类型']=='Condition'])
print(f"\n=== Decorator 节点 ({len(df[df['类型']=='Decorator'])}) ===")
display(df[df['类型']=='Decorator'])

## 3. 扫描工作空间查找对应 HPP 和 CPP 文件

在 `rm_behavior_tree` 包中搜索与每个节点 ID 匹配的源文件（PascalCase → snake_case 转换）。

In [ ]:
from pathlib import Path
import re

def pascal_to_snake(name):
    """将 PascalCase 转换为 snake_case"""
    s = re.sub(r'(?<=[a-z0-9])([A-Z])', r'_\1', name)
    s = re.sub(r'(?<=[A-Z])([A-Z][a-z])', r'_\1', s)
    return s.lower()

base_dir = Path(r"d:\open_resource_project\ros2_ws\src\rm_behavior_tree\rm_behavior_tree")
include_dir = base_dir / "include"
plugins_dir = base_dir / "plugins"

# 搜索所有 hpp 和 cpp 文件
all_hpp = list(include_dir.rglob("*.hpp"))
all_cpp = list(plugins_dir.rglob("*.cpp"))
# 也搜索 src 目录
all_cpp += list((base_dir / "src").rglob("*.cpp"))

file_match = {}
for nid in all_nodes:
    snake = pascal_to_snake(nid)
    hpp_found = [f for f in all_hpp if f.stem == snake]
    cpp_found = [f for f in all_cpp if f.stem == snake]
    file_match[nid] = {
        "hpp": hpp_found[0].relative_to(base_dir) if hpp_found else None,
        "cpp": cpp_found[0].relative_to(base_dir) if cpp_found else None,
        "snake_name": snake
    }

# 展示结果
rows = []
for nid, match in file_match.items():
    rows.append({
        "节点ID": nid,
        "snake_case名": match["snake_name"],
        "HPP文件": str(match["hpp"]) if match["hpp"] else "❌ 未找到",
        "CPP文件": str(match["cpp"]) if match["cpp"] else "❌ 未找到"
    })

df_files = pd.DataFrame(rows)
display(df_files)

# 统计
not_found_hpp = df_files[df_files["HPP文件"].str.contains("未找到")]
not_found_cpp = df_files[df_files["CPP文件"].str.contains("未找到")]
print(f"\n⚠️  未找到 HPP 文件的节点 ({len(not_found_hpp)}):")
for _, r in not_found_hpp.iterrows():
    print(f"  - {r['节点ID']} (尝试文件名: {r['snake_case名']}.hpp)")
print(f"\n⚠️  未找到 CPP 文件的节点 ({len(not_found_cpp)}):")
for _, r in not_found_cpp.iterrows():
    print(f"  - {r['节点ID']} (尝试文件名: {r['snake_case名']}.cpp)")

## 4. Action 节点源码解析与用法说明

### 4.1 数据订阅类 Action 节点

这些节点负责从 ROS2 话题订阅数据并写入行为树黑板。

---

#### `SubGameStatus`
- **源文件**: `plugins/action/sub_game_status.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::GameStatus>`
- **功能**: 订阅 `/game_status` 话题，获取比赛状态（比赛阶段、剩余时间等）
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /game_status | 话题名 |
  | output | game_status | GameStatus | {game_status} | 比赛状态消息 |
- **XML用法**:
```xml
<SubGameStatus topic_name="/game_status" game_status="{game_status}"/>
```
- **✅ 已修复**: include guard 从 `SUB_ALL_ROBOT_HP_HPP_` 修正为 `SUB_GAME_STATUS_HPP_`

---

#### `SubRobotStatus`
- **源文件**: `plugins/action/sub_robot_status.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::RobotStatus>`
- **功能**: 订阅 `/robot_status` 话题，获取机器人状态（血量、射击热量、被攻击等）
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /robot_status | 话题名 |
  | output | robot_status | shared_ptr&lt;RobotStatus&gt; | {robot_status} | 机器人状态（注意是 shared_ptr） |
- **特殊说明**: 输出类型为 `std::shared_ptr<RobotStatus>`，下游条件节点（IsHPBelow、IsHPAbove等）需要使用相同的 shared_ptr 类型读取。

---

#### `SubArmors`
- **源文件**: `plugins/action/sub_armors.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<auto_aim_interfaces::msg::Armors>`
- **功能**: 订阅装甲板检测结果话题
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /detector/armors | 检测话题名 |
  | output | armors | Armors | {armors} | 装甲板检测消息 |

---

#### `SubAllRobotHP`
- **源文件**: `plugins/action/sub_all_robot_hp.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::AllRobotHP>`
- **功能**: 订阅全体机器人血量话题
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /robot_hp | 话题名 |
  | output | robot_hp | AllRobotHP | {robot_hp} | 全体血量消息 |

---

#### `SubDecisionNum`
- **源文件**: `plugins/action/sub_decision_num.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::DecisionNum>`
- **功能**: 订阅决策编号话题（用于模式切换）
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /decision_num | 话题名 |
  | output | decision_num | DecisionNum | {decision_num} | 决策编号消息 |

---

#### `SubRFIDStatus`
- **源文件**: `plugins/action/sub_rfid_status.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::RFID>`
- **功能**: 订阅 RFID 状态话题（用于补给区刷卡判定）
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /rfid_status | 话题名 |
  | output | rfid_status | RFID | {rfid.status} | RFID 消息（含 rfid_supply_arrived 字段） |

---

#### `SubRobotPosition`
- **源文件**: `plugins/action/sub_robot_position.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`（自建订阅，非 RosTopicSubNode）
- **功能**: 订阅机器人位置话题（x, y, yaw），通过 `setOutput()` 在 `tick()` 中写入端口
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | topic_name | string | /red_standard_robot1 | 位置话题名 |
  | output | pose_x | double | - | X 坐标 |
  | output | pose_y | double | - | Y 坐标 |
  | output | pose_yaw | double | - | 朝向角 |
- **✅ 已修复**: 移除了回调中直接 `bb->set()` 写黑板的代码，改为仅通过 `tick()` 中 `setOutput()` 写入（有 mutex 保护，线程安全）。

### 4.2 控制输出类 Action 节点

这些节点负责向 ROS2 话题发布控制指令。

---

#### `SendGoal`
- **源文件**: `plugins/action/send_goal.hpp` / `.cpp`
- **基类**: `BT::RosTopicPubNode<geometry_msgs::msg::PoseStamped>`
- **功能**: 发布导航目标点（PoseStamped）到 `goal_pose` 话题
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | goal_pose | PoseStamped | {goal_pose} | 完整目标位姿（优先） |
  | input | goal_x | double | 0.0 | 目标 X 坐标（备选） |
  | input | goal_y | double | 0.0 | 目标 Y 坐标（备选） |
  | input | action_name | string | navigate_to_pose | 兼容字段（实际不用于 action） |
  | input | min_interval_ms | int | 0 | 发布最小间隔（ms），0=每次 tick 都发布 |
- **逻辑**: 
  1. 优先读取 `goal_pose`，若无则用 `goal_x`/`goal_y` 构建
  2. 支持节流（`min_interval_ms`）：相同目标在间隔内不重复发布
  3. header.frame_id 使用 `"map"` 坐标系
- **✅ 已修复**: `frame_id` 从 `"chassis"` 改为 `"map"`
- **XML用法**:
```xml
<SendGoal name="GoSupplyToHeal"
          goal_x="{ally_supply_pose_x}"
          goal_y="{ally_supply_pose_y}"
          action_name="navigate_to_pose"/>
```

---

#### `RobotControl`
- **源文件**: `plugins/action/robot_control.hpp` / `.cpp`
- **基类**: `BT::RosTopicPubNode<rm_decision_interfaces::msg::RobotControl>`
- **功能**: 发布机器人云台/底盘控制指令
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | stop_gimbal_scan | bool | False | 是否停止云台扫描 |
  | input | chassis_spin | bool | False | 是否启用底盘旋转 |
- **XML用法**:
```xml
<RobotControl stop_gimbal_scan="False" chassis_spin="True"/>
```

---

#### `NavControlCmd`
- **源文件**: `plugins/action/nav_control_cmd.hpp` / `.cpp`
- **基类**: `BT::RosTopicPubNode<rm_decision_interfaces::msg::NavControlCmd>`
- **功能**: 发布导航控制命令（命令类型、紧急停止）
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | cmd_type | int | 0 | 命令类型 |
  | input | emergency_stop | bool | False | 紧急停止 |

---

#### `CancelNavGoal`
- **源文件**: `plugins/action/cancel_nav_goal.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`
- **功能**: 调用 Nav2 的 `_action/cancel_goal` 服务取消导航目标
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | action_name | string | navigate_to_pose | action 命名空间 |
  | input | min_interval_ms | int | 0 | 防止频繁 cancel |
  | input | wait_for_service_ms | int | 0 | 等待服务（0=不等待） |
  | input | fail_on_unavailable | bool | false | 服务不可用时是否返回 FAILURE |
  | input | wait_response_ms | int | 0 | 等待响应（0=fire-and-forget） |
  | output | return_code | int | - | cancel 返回码 |
- **XML用法**:
```xml
<CancelNavGoal action_name="navigate_to_pose"/>
```

### 4.3 工具/逻辑类 Action 节点

---

#### `GetCurrentLocation`
- **源文件**: `plugins/action/get_current_location.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`
- **功能**: 通过 TF2 查找 `map` → `gimbal_yaw` 变换获取当前位置
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | output | current_location | TransformStamped | {current_location} | 当前位姿变换 |
- **⚠️ 设计备注**: 构造函数中自建 `rclcpp::Node` + TF listener。`TransformListener` 默认 `spin_thread=true` 会自行创建回调线程，因此 TF 缓冲实际可以更新。后续优化可考虑复用 `BT::RosNodeParams` 的共享节点。

---

#### `MoveAround`
- **源文件**: `plugins/action/move_around.hpp` / `.cpp`
- **基类**: `BT::StatefulActionNode` + `rclcpp::Node`（多继承）
- **功能**: 以当前位置为圆心，在给定半径内随机生成目标点并发送
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | expected_nearby_goal_count | int | 5 | 生成随机点数量 |
  | input | expected_dis | float | 0.3 | 随机半径（米） |
  | input | message | TransformStamped | {current_location} | 当前位置 |
- **✅ 已修复（Bug#3 阻塞问题）**: 
  - 移除 `std::this_thread::sleep_for(1000ms)` 阻塞调用
  - 添加 `last_goal_time_` 成员变量，改用时间戳比较实现非阻塞等待
  - `onStart()` 初始化时间戳（-1000ms偏移保证首次立即发送）
  - `onRunning()` 检查 `elapsed < 1000ms` 时直接返回 RUNNING（不阻塞BT线程）
- **⚠️ 设计备注**: 多继承 `rclcpp::Node` 但未接入 executor。rclcpp publisher 在无 executor 时仍能发送消息（DDS 直接发送），实际影响有限。

---

#### `InitBlackboardConfig`
- **源文件**: `plugins/action/init_blackboard_config.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`
- **功能**: 初始化行为树配置参数到黑板（一次性写入）
- **端口**（全部 output）:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | heal_wait_ms | uint64 | 2000 | 等待回血时间门限（ms） |
  | heal_min_ratio | double | 0.60 | 最低回血比例 |
  | search_timeout_ms | uint64 | 3000 | 搜卡超时（ms） |
  | recovery_timeout_ms | uint64 | 15000 | 恢复超时（ms） |
  | supply_goal_x | double | 0.0 | 补给区 X 坐标 |
  | supply_goal_y | double | 0.0 | 补给区 Y 坐标 |
  | arrive_radius | double | 0.6 | 到达判定半径 |
- **注意**: 只在首次 tick 时写入（`initialized_` 标志），后续直接返回 SUCCESS。

---

#### `SetNavGoalFromConfig`
- **源文件**: `plugins/action/set_nav_goal_from_config.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`
- **功能**: 从配置读取目标坐标，安全限幅后写入黑板
- **端口**:
  | 方向 | 名称 | 类型 | 默认值 | 说明 |
  |------|------|------|--------|------|
  | input | cfg_x | double | {cfg.supply_goal_x} | 配置X |
  | input | cfg_y | double | {cfg.supply_goal_y} | 配置Y |
  | output | goal_x | double | {nav.goal_x} | 导航目标X |
  | output | goal_y | double | {nav.goal_y} | 导航目标Y |
  | output | goal_pose | PoseStamped | - | 完整目标位姿 |
- **防护**: NaN/inf 检查、±50m 限幅、历史目标缓存

---

#### `DetectRespawnAndSetRecovery`
- **源文件**: `plugins/action/detect_respawn_and_set_recovery.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::RobotStatus>`
- **功能**: 检测战亡→复活的上升沿，设置恢复标志位
- **端口**（双向 inout 为主）:
  | 方向 | 名称 | 类型 | 说明 |
  |------|------|------|------|
  | inout | hp_cur | int | 当前血量 |
  | inout | was_dead | bool | 上一帧是否死亡 |
  | inout | need_recovery | bool | 是否需要恢复 |
  | inout | recovery_start_ms | uint64 | 恢复开始时间戳 |
  | inout | search_start_ms | uint64 | 搜卡开始时间戳 |
  | inout | heal_start_ms | uint64 | 回血开始时间戳 |
- **特性**: 连续帧防抖（2帧稳定存活）、复活锁（5秒）、fallback 订阅机制

---

#### `ClearRecoveryFlag`
- **源文件**: `plugins/action/clear_recovery_flag.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`
- **功能**: 清除恢复标志和计时器状态
- **逻辑**: 将 `need_recovery=false`，`heal_start_ms=0`，`search_start_ms=0`，`recovery_start_ms=0`

---

#### `InitSearchTimerIfNeeded`
- **源文件**: `plugins/action/init_search_timer_if_needed.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`
- **功能**: 初始化搜卡计时（仅当 `search_start_ms == 0` 时写入当前时间戳）

---

#### `MicroSearchSupplyCard`
- **源文件**: `plugins/action/micro_search_supply_card.hpp` / `.cpp`
- **基类**: `BT::StatefulActionNode`
- **功能**: 到达补给区附近后小范围十字移动以刷 RFID 卡
- **策略**: 以底盘当前位置为中心，十字四方向微移（±12cm），超时扩圈（每次+6cm，最大35cm）
- **端口**:
  | 方向 | 名称 | 类型 | 说明 |
  |------|------|------|------|
  | inout | search_start_ms | uint64 | 搜卡开始时间 |
  | input | timeout_ms | int | 搜卡超时 |
  | input | rfid_supply_arrived | bool | 是否已刷到补给区卡 |
  | input | rfid_status | RFID | RFID 消息（备用检查） |

---

#### `WaitAndHeal`
- **源文件**: `plugins/action/wait_and_heal.hpp` / `.cpp`
- **基类**: `BT::RosTopicSubNode<rm_decision_interfaces::msg::RobotStatus>`
- **功能**: 在补给区等待回血，同时监听 RobotStatus 更新血量
- **判定条件**: `已等待时间 >= heal_wait_ms` **且** `当前血量比例 >= heal_min_ratio` → SUCCESS
- **端口**:
  | 方向 | 名称 | 类型 | 说明 |
  |------|------|------|------|
  | inout | heal_start_ms | uint64 | 回血开始时间戳 |
  | input | heal_wait_ms | uint64 | 等待时间门限 |
  | input | heal_min_ratio | double | 最低回血比例 |
  | input | hp_cur / hp_max / now_ms | - | 兼容端口（实际从 ROS 获取） |

---

#### `KeepRunning`
- **源文件**: `plugins/action/keep_running.hpp` / `.cpp`
- **基类**: `BT::StatefulActionNode`
- **功能**: 永远返回 `RUNNING`
- **用途**: 配合条件节点阻塞分支（如 `NotArrived` + `KeepRunning`）
- **✅ 已修复（Bug#2）**: 基类从 `SyncActionNode` 改为 `StatefulActionNode`，实现 `onStart()/onRunning()` 返回 RUNNING，`onHalted()` 空实现。SyncActionNode 设计上不允许返回 RUNNING，现已符合 BehaviorTree.CPP v4 规范。

## 5. Condition 节点源码解析与用法说明

条件节点返回 `SUCCESS`（条件成立）或 `FAILURE`（条件不成立），用于行为树的分支判断。

---

#### `IsGameTime`
- **源文件**: `plugins/condition/is_game_time.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断当前是否处于指定比赛阶段和时间范围
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | message | GameStatus | {game_status} | 比赛状态 |
  | game_progress | int | 4 | 期望阶段（4=比赛中） |
  | lower_remain_time | int | 0 | 剩余时间下限 |
  | higher_remain_time | int | 300 | 剩余时间上限 |
- **返回**:
  - SUCCESS: `game_progress 匹配` 且 `lower ≤ remain_time ≤ higher`
  - FAILURE: 不满足或消息缺失
- **比赛阶段编码**: 0=未开始, 1=准备, 2=自检, 3=倒计时, 4=比赛中, 5=结算

---

#### `IsHPBelow`
- **源文件**: `plugins/condition/is_hp_below.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断当前血量 < 阈值
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | message | shared_ptr&lt;RobotStatus&gt; | {robot_status} | 机器人状态 |
  | hp_threshold | int | 100 | 血量阈值 |
- **返回**: SUCCESS 当 `current_hp < hp_threshold`
- **XML用法**:
```xml
<IsHPBelow message="{robot_status}" hp_threshold="200"/>
```

---

#### `IsHPAbove`
- **源文件**: `plugins/condition/is_hp_above.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断当前血量 >= 阈值
- **返回**: SUCCESS 当 `current_hp >= hp_threshold`

---

#### `IsDead`
- **源文件**: `plugins/condition/is_dead.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断机器人是否死亡（`current_hp <= 0`）
- **端口**: `message` = `shared_ptr<RobotStatus>`
- **返回**: SUCCESS 当 `current_hp <= 0`

---

#### `IsDetectEnemy`
- **源文件**: `plugins/condition/is_detect_enemy.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断是否检测到敌人（带数据时间戳过期检查）
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | message | auto_aim_interfaces::msg::RMUL | {armors} | 检测结果 |
- **逻辑**:
  1. 检查消息时间戳与当前时间差是否 ≤ 100ms
  2. 过期数据直接返回 FAILURE
  3. 未过期时检查 `is_detect_enemy` 字段
- **⚠️ 待确认**: 代码中输入类型为 `RMUL`，而 `SubArmors` 输出的是 `Armors` 类型。需确认 `IsDetectEnemy.xml` 子树中是否有独立的 RMUL 订阅做类型转换。

---

#### `IsAttacked` ✅ 已修复拼写
- **源文件**: `plugins/condition/is_attacked.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断机器人是否正在被攻击
- **返回**: SUCCESS 当 `is_attacked == true`
- **✅ 已修复（Bug#1）**: 类名 `IsAttakedAction`→`IsAttackedAction`，工厂注册名 `"IsAttaked"`→`"IsAttacked"`，所有16个XML文件同步更新

---

#### `IsFriendOK`
- **源文件**: `plugins/condition/is_friend_ok.hpp` / `.cpp`
- **基类**: `BT::SimpleConditionNode`
- **功能**: 判断我方平均血量是否高于敌方平均血量
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | message | AllRobotHP | {robot_hp} | 全体血量 |
  | friend_color | string | red | 我方颜色 |
- **逻辑**: 统计 `1号(英雄) + 3号(步兵) + 4号(步兵) + 7号(哨兵)` 的平均血量进行比较

---

#### `IsStatusOK` ✅ 已修复端口声明
- **源文件**: `plugins/condition/is_status_ok.hpp` / `.cpp`
- **基类**: `BT::ConditionNode`
- **功能**: 综合状态检查（血量、前哨站、热量）
- **端口**（已与 C++ `providedPorts()` 对齐）:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | message | shared_ptr&lt;RobotStatus&gt; | {robot_status} | 机器人状态 |
  | hp_threshold | int | 0 | 血量阈值 |
  | blue_outpost_hp_threshold | int | 0 | 蓝方前哨站HP阈值 |
  | red_outpost_hp_threshold | int | 0 | 红方前哨站HP阈值 |
  | heat_threshold | int | 9999 | 热量阈值 |
- **返回**: SUCCESS 当所有条件满足：`hp >= 阈值` 且 `前哨站HP >= 阈值` 且 `热量 <= 阈值`
- **✅ 已修复（Bug#15）**: XML端口从旧的 `sentry_hp` 等更正为 C++ 代码中实际的端口名

---

#### `IsRecoveryNeeded`
- **源文件**: `plugins/condition/is_recovery_needed.hpp` / `.cpp`
- **基类**: `BT::ConditionNode`
- **功能**: 直接读取黑板 `need_recovery` 标志
- **返回**: SUCCESS 当 `need_recovery == true`

---

#### `IsSupplyCardDetected`
- **源文件**: `plugins/condition/is_supply_card_detected.hpp` / `.cpp`
- **基类**: `BT::ConditionNode`
- **功能**: 判断补给区 RFID 是否已被刷到
- **返回**: SUCCESS 当 `rfid_supply_arrived == true`

---

#### `IsWithinScope`
- **源文件**: `plugins/condition/is_within_scope.hpp` / `.cpp`
- **基类**: `BT::ConditionNode`
- **功能**: 判断机器人是否在目标点的有效半径内
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | pose_x / pose_y | double | {pose.x} / {pose.y} | 当前位置 |
  | goal_x / goal_y | double | {cfg.supply_goal_x/y} | 目标位置 |
  | arrive_radius | double | {cfg.arrive_radius} | 有效半径 |
- **返回**: SUCCESS 当 `distance(pose, goal) ≤ arrive_radius`
- **⚠️ 配置备注**: 默认目标坐标 (0,0) 是 TODO 占位，需通过 `InitBlackboardConfig` 注入实际补给区坐标。

---

#### `NotArrived` ✅ 已修复端口
- **源文件**: `plugins/condition/not_arrived.hpp` / `.cpp`
- **基类**: `BT::SyncActionNode`（注意：不是 ConditionNode）
- **功能**: 判断是否**未**到达补给区（语义反转的条件判断）
- **端口**: `rfid_status` (RFID msg)
- **返回**: SUCCESS 当 `rfid_supply_arrived == false`（未到达）
- **✅ 已修复（Bug#8）**: XML端口从 `arrived`(bool) 改为 `rfid_status`(RFID msg)，与 C++ `providedPorts()` 一致

## 6. Decorator / Control 节点源码解析与用法说明

---

#### `RateController` (Decorator) ✅ 已修复动态频率
- **源文件**: `plugins/decorator/rate_controller.hpp` / `.cpp`
- **基类**: `BT::DecoratorNode`
- **功能**: 以指定频率（hz）tick 子节点，控制子节点执行速率
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | hz | double | 10.0 | 控制频率（支持运行时动态修改） |
- **逻辑**:
  1. 从 IDLE 状态开始时重置计时器
  2. 首次 tick 或计时器到期时执行子节点
  3. 子节点处于 RUNNING 状态时每次都继续 tick
  4. 子节点 SUCCESS 时重置计时器
- **XML用法**:
```xml
<RateController hz="1">
    <SendGoal goal_x="1.0" goal_y="2.0" action_name="navigate_to_pose"/>
</RateController>
```
- **✅ 已修复（Bug#10）**: 在 `tick()` 开头添加动态 hz 重读逻辑，每次 tick 都会重新从端口读取 `hz` 值并更新 `period_`，支持运行时动态修改频率。

---

#### `DecisionSwitch` (Control) ✅ 已修复越界检查
- **源文件**: `plugins/control/decision_switch.hpp` / `.cpp`
- **基类**: `BT::ControlNode`
- **功能**: 根据 `decision_num` 值切换执行不同子节点（1→child[0], 2→child[1]）
- **端口**:
  | 名称 | 类型 | 默认值 | 说明 |
  |------|------|--------|------|
  | decision_num | DecisionNum | {decision_num} | 决策编号 |
- **✅ 已修复（Bug#11）**: 添加了 `children_nodes_.size()` 边界检查，子节点数量不足时打印警告并返回 FAILURE，避免越界崩溃

---

### 补充节点（存在于源码但未出现在当前 XML 中）

| 节点ID | 类型 | 说明 |
|--------|------|------|
| `SentryFollower` | Action | 跟踪敌方装甲板位置，发布追踪目标（TF变换） |
| `PrintMessage` | Action | 调试用：打印消息到终端和 ROS 日志 |
| `IsOutpostOK` | Condition | 判断前哨站是否存活 |
| `NavigateToGoal` | Condition | RFID 到达判断（与 IsSupplyCardDetected 功能重复） |

## 7. SubTree 引用的子树 XML 文件检查

检查主 XML 中 `<include>` 引用的所有子树文件是否存在。

In [ ]:
config_dir = Path(r"d:\open_resource_project\ros2_ws\src\rm_behavior_tree\rm_behavior_tree\config")

# 提取所有 include 的子树文件
includes = root.findall("include")
print("=== SubTree 引用的子树 XML 文件 ===\n")
for inc in includes:
    path = inc.get("path")
    full_path = config_dir / path
    exists = full_path.exists()
    status = "✅ 存在" if exists else "❌ 不存在"
    print(f"  {path:40s} → {status}")
    
    # 如果存在，检查其使用的自定义节点
    if exists:
        try:
            sub_tree = ET.parse(full_path)
            sub_root = sub_tree.getroot()
            # 查找所有使用的节点
            used_nodes = set()
            for elem in sub_root.iter():
                tag = elem.tag
                if tag not in ["root", "BehaviorTree", "include", "TreeNodesModel",
                               "Sequence", "Fallback", "ReactiveFallback", "ReactiveSequence",
                               "WhileDoElse", "SubTree", "ForceSuccess", "ForceFailure",
                               "Inverter", "Repeat", "Parallel", "IfThenElse",
                               "input_port", "output_port", "inout_port",
                               "Action", "Condition", "Decorator", "Control"]:
                    used_nodes.add(tag)
            if used_nodes:
                # 检查是否在 TreeNodesModel 中定义
                missing = used_nodes - set(all_nodes.keys())
                defined_in_model = used_nodes & set(all_nodes.keys())
                if defined_in_model:
                    print(f"    使用的已定义节点: {', '.join(sorted(defined_in_model))}")
                if missing:
                    print(f"    ⚠️ 未在 TreeNodesModel 中定义的节点: {', '.join(sorted(missing))}")
        except Exception as e:
            print(f"    解析失败: {e}")

## 8. 黑板端口连接一致性检测

分析主树 XML 中黑板变量的读写关系。

In [ ]:
import re

# 分析 TreeNodesModel 中所有变量的读写关系
writers = {}  # 黑板变量 -> 写入节点列表
readers = {}  # 黑板变量 -> 读取节点列表

bb_var_pattern = re.compile(r'\{(.+?)\}')

for nid, info in all_nodes.items():
    for p in info["ports"]["output"]:
        match = bb_var_pattern.search(p["default"])
        if match:
            var = match.group(1)
            writers.setdefault(var, []).append(nid)
    
    for p in info["ports"]["input"]:
        match = bb_var_pattern.search(p["default"])
        if match:
            var = match.group(1)
            readers.setdefault(var, []).append(nid)
    
    for p in info["ports"]["inout"]:
        match = bb_var_pattern.search(p["default"])
        if match:
            var = match.group(1)
            writers.setdefault(var, []).append(f"{nid}(inout)")
            readers.setdefault(var, []).append(f"{nid}(inout)")

# 汇总
all_vars = set(writers.keys()) | set(readers.keys())
print("=== 黑板变量读写一致性检查 ===\n")

orphan_write = []
orphan_read = []

for var in sorted(all_vars):
    w = writers.get(var, [])
    r = readers.get(var, [])
    
    if w and not r:
        orphan_write.append(var)
    elif r and not w:
        orphan_read.append(var)

if orphan_write:
    print("⚠️ 只有写入、从未被读取的黑板变量：")
    for var in orphan_write:
        print(f"  - {var:30s} ← 写入者: {', '.join(writers[var])}")

if orphan_read:
    print("\n⚠️ 只有读取、从未被写入的黑板变量（可能由子树或外部注入）：")
    for var in orphan_read:
        print(f"  - {var:30s} → 读取者: {', '.join(readers[var])}")

print("\n=== 正常连接的黑板变量 ===")
for var in sorted(all_vars):
    w = writers.get(var, [])
    r = readers.get(var, [])
    if w and r:
        print(f"  {var:30s} | 写: {', '.join(w):40s} → 读: {', '.join(r)}")

## 9. 🔴 发现的逻辑错误与潜在问题汇总

以下是通过源码审查发现的所有问题及修复记录：

| # | 严重度 | 节点/位置 | 问题描述 | 修复状态 | 修复过程 |
|---|--------|-----------|----------|----------|----------|
| 1 | 🟡 低 | `IsAttaked` | **拼写错误**：类名 `IsAttakedAction`、节点ID `IsAttaked`，应为 `IsAttacked` | ✅ 已修复 | hpp: `IsAttakedAction`→`IsAttackedAction`；cpp: 类名+工厂注册名全部替换；**16个XML文件**批量替换 `IsAttaked`→`IsAttacked` |
| 2 | 🔴 高 | `KeepRunning` | 继承 `BT::SyncActionNode` 但返回 `RUNNING`，SyncActionNode 不允许返回 RUNNING | ✅ 已修复 | hpp: 基类 `SyncActionNode`→`StatefulActionNode`，`tick()`→`onStart()/onRunning()/onHalted()`；cpp: 构造函数基类替换，实现三个新方法（onStart/onRunning 返回 RUNNING，onHalted 空） |
| 3 | 🔴 高 | `MoveAround` | `onRunning()` 使用 `sleep_for(1000ms)` **阻塞行为树线程** | ✅ 已修复 | hpp: 添加 `last_goal_time_` 成员变量；cpp: 移除 `sleep_for`，改用时间戳比较（`elapsed < 1000` 则直接返回 RUNNING），`onStart()` 初始化时间戳为 -1000ms 保证首次立即执行 |
| 4 | 🟡 中 | `MoveAround` | 多继承 `rclcpp::Node` 但未被 executor spin，话题发布可能不工作 | ⚠️ 未修复 | 设计层面问题。rclcpp publisher 在无 executor 时仍能 publish（DDS 直接发送），实际影响有限。彻底修复需改构造函数，风险较大暂不处理 |
| 5 | 🟡 中 | `GetCurrentLocation` | 自建 `rclcpp::Node` + TF listener，无 executor spin，TF buffer 可能无法更新 | ⚠️ 未修复 | `TransformListener` 构造时默认 `spin_thread=true` 会自行创建回调线程，因此 TF 缓冲实际可以更新。属于设计层面可优化项 |
| 6 | 🔴 高 | `SubGameStatus` hpp | **include guard 冲突**：使用了 `SUB_ALL_ROBOT_HP_HPP_` 的 guard（复制粘贴错误） | ✅ 已修复 | `#ifndef/#define` 从 `SUB_ALL_ROBOT_HP_HPP_` 改为 `SUB_GAME_STATUS_HPP_`，`#endif` 注释同步修改 |
| 7 | 🟡 中 | `IsDetectEnemy` | 输入类型 `RMUL` vs `SubArmors` 输出 `Armors` 类型不匹配 | ⚠️ 未修复 | 需确认 `IsDetectEnemy.xml` 子树内部是否有独立的 RMUL 订阅节点做类型转换，需结合子树分析 |
| 8 | 🟡 中 | `NotArrived` | XML 端口 `arrived`(bool) vs C++ 端口 `rfid_status`(RFID msg) 不匹配 | ✅ 已修复 | 3v3_new.xml 中 `<input_port name="arrived">` 改为 `<input_port name="rfid_status" default="{rfid.status}"/>`，与 C++ `providedPorts()` 一致 |
| 9 | 🟡 低 | `SubRobotPosition` | 回调中 `bb->set()` 直接写黑板，绕过端口系统，存在线程安全隐患 | ✅ 已修复 | cpp: 移除回调中整个 `bb->set("pose.x/y/yaw")` 代码块（含 try-catch），数据仅通过 `tick()` 中 `setOutput()` 写入（已有 mutex 保护） |
| 10 | 🟡 低 | `RateController` | `hz` 仅构造时读取，运行时无法动态修改频率 | ✅ 已修复 | cpp `tick()` 开头添加：`double hz=1.0; getInput("hz",hz); if(hz>0) period_=1.0/hz;` 每次 tick 重新读取 hz 端口 |
| 11 | 🟡 中 | `DecisionSwitch` | `children_nodes_[0]/[1]` 无边界检查，子节点不足时越界崩溃 | ✅ 已修复 | cpp: 添加 `const size_t num_children = children_nodes_.size();`，case 1/2 前各加 `if(num_children < N)` 检查，不足时打印警告并返回 FAILURE |
| 12 | 🟡 低 | `SendGoal` | `frame_id` 硬编码 `"chassis"`，导航目标应使用 `"map"` 坐标系 | ✅ 已修复 | cpp: `msg.header.frame_id = "chassis"` → `msg.header.frame_id = "map"` |
| 13 | 🟡 低 | 主树 XML Home | else 分支 Home `goal_x/y="0.0"` 硬编码，标注 TODO | ⚠️ 未修复 | 属于配置层面，需根据实际场地设置出生点坐标，不适合在代码审查中强行修改 |
| 14 | 🟡 低 | `IsWithinScope` | 默认目标坐标 (0,0) 标注 TODO，补给区坐标未设置 | ⚠️ 未修复 | 同上，需通过 `InitBlackboardConfig` 注入实际坐标，属于部署配置项 |
| 15 | 🟡 中 | `IsStatusOK` XML | XML 声明 `sentry_hp` 端口，C++ 实际为 `hp_threshold/blue_outpost_hp_threshold/red_outpost_hp_threshold/heat_threshold` | ✅ 已修复 | 3v3_new.xml 中 IsStatusOK 端口声明完全重写，与 C++ `providedPorts()` 对齐：`hp_threshold=0, blue_outpost_hp_threshold=0, red_outpost_hp_threshold=0, heat_threshold=9999` |
| 16 | 🟡 中 | XML 缺失节点 | 12个节点在 TreeNodesModel 中声明但无源文件 | ✅ 已修复 | 3v3_new.xml 中注释掉 12 个无源文件的节点声明：Rotate、ScanStatus、SetGoal、SubHP、SelectBattleMode、RefSerialBlackboardSync、StopMotion、UpdateWasDead、RecoveryTimeoutGuard、IsGameStart、IsModeRight、IsModeLeft |

### 修复统计
- **已修复**: 11/16 (Bug #1,2,3,6,8,9,10,11,12,15,16)
- **未修复(设计层面)**: 5/16 (Bug #4,5,7,13,14) — 风险低，需设计决策或部署配置

## 10. 自动生成节点速查表

汇总所有自定义节点信息并导出为 CSV。

In [ ]:
# 功能简述映射
func_desc = {
    "GetCurrentLocation": "通过TF2获取map→gimbal_yaw变换，输出当前位姿",
    "IsDetectEnemy": "判断RMUL消息中是否检测到敌人（含时间戳过期验证）",
    "MoveAround": "以当前位置为圆心随机生成目标点进行小范围移动",
    "Rotate": "旋转底盘（未找到源文件）",
    "ScanStatus": "设置扫描状态（未找到源文件）",
    "SendGoal": "发布PoseStamped导航目标到goal_pose话题",
    "SetGoal": "设置导航目标（未找到源文件）",
    "SubAllRobotHP": "订阅/robot_hp话题获取全体机器人血量",
    "SubArmors": "订阅/detector/armors话题获取装甲板检测结果",
    "SubGameStatus": "订阅/game_status话题获取比赛状态",
    "SubHP": "订阅/sentry_hp话题获取哨兵血量（未找到源文件）",
    "SubRobotStatus": "订阅/robot_status获取本机状态(shared_ptr)",
    "SelectBattleMode": "根据状态选择战斗模式（未找到源文件）",
    "SubDecisionNum": "订阅/decision_num获取决策编号",
    "SubRFIDStatus": "订阅/rfid_status获取RFID补给区状态",
    "InitBlackboardConfig": "一次性初始化配置参数到黑板(cfg.*)",
    "SubRobotPosition": "订阅位置话题写入pose_x/y/yaw到黑板",
    "RefSerialBlackboardSync": "串口数据同步到黑板（未找到源文件）",
    "StopMotion": "停止运动（未找到源文件）",
    "KeepRunning": "永远返回RUNNING（阻塞用）",
    "CancelNavGoal": "调用cancel_goal服务取消Nav2导航",
    "UpdateWasDead": "更新死亡状态标志（未找到源文件）",
    "DetectRespawnAndSetRecovery": "检测复活上升沿，设置恢复标志(防抖)",
    "ClearRecoveryFlag": "清除恢复标志和计时器状态",
    "SetNavGoalFromConfig": "从cfg_x/y读取并安全限幅后写入goal",
    "InitSearchTimerIfNeeded": "首次初始化搜卡计时(search_start_ms)",
    "MicroSearchSupplyCard": "补给区十字微移搜索RFID卡(超时扩圈)",
    "WaitAndHeal": "补给区等待回血(时间门+血量门双判定)",
    "RecoveryTimeoutGuard": "恢复超时守卫（未找到源文件）",
    "RobotControl": "发布云台/底盘控制指令",
    "NavControlCmd": "发布导航控制命令(cmd_type/emergency_stop)",
    "IsAttaked": "判断机器人是否被攻击(拼写错误:应为IsAttacked)",
    "IsFriendOK": "比较双方平均血量判断我方血量优势",
    "IsGameStart": "判断比赛是否开始（可能是IsGameTime的旧版）",
    "IsGameTime": "判断比赛阶段+剩余时间是否在区间内",
    "IsStatusOK": "综合判断HP/前哨站/热量多项状态",
    "IsHPAbove": "判断 current_hp >= threshold",
    "IsHPBelow": "判断 current_hp < threshold",
    "IsModeRight": "判断决策模式是否为右侧（未找到源文件）",
    "IsModeLeft": "判断决策模式是否为左侧（未找到源文件）",
    "IsDead": "判断 current_hp <= 0",
    "IsRecoveryNeeded": "读取黑板need_recovery标志",
    "IsSupplyCardDetected": "判断RFID补给区卡是否已刷到",
    "IsWithinScope": "判断机器人是否在目标点有效半径内",
    "NotArrived": "判断是否未到达补给区(rfid反转语义)",
    "RateController": "装饰器:按指定hz频率tick子节点"
}

# 生成完整速查表
rows = []
for nid, info in all_nodes.items():
    match = file_match.get(nid, {})
    all_port_names = []
    for kind in ["input", "output", "inout"]:
        for p in info["ports"][kind]:
            all_port_names.append(f"{p['name']}({kind[0]})")
    
    rows.append({
        "节点ID": nid,
        "类型": info["type"],
        "端口列表": ", ".join(all_port_names) if all_port_names else "(无端口)",
        "功能简述": func_desc.get(nid, "未知"),
        "HPP文件": str(match.get("hpp", "未找到")),
        "CPP文件": str(match.get("cpp", "未找到"))
    })

df_summary = pd.DataFrame(rows).sort_values(["类型", "节点ID"]).reset_index(drop=True)
display(df_summary)

# 导出CSV
csv_path = config_dir / "bt_nodes_reference.csv"
df_summary.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n✅ 速查表已导出到: {csv_path}")

---

## 附录：主行为树结构概览

```
MainBehaviorTree (Sequence)
├── PerceptionAndBlackboard (SubTree) — 订阅所有话题数据写入黑板
└── WhileDoElse
    ├── [条件] IsGameTime: 比赛进行中？(game_progress=4, 0~300s)
    ├── [满足] Fallback
    │   ├── IsDeadAndDispelDebuff (SubTree) — 战亡检测与虚弱解除
    │   └── Fallback
    │       ├── IsDetectEnemy (SubTree) — 发现敌人→战斗逻辑
    │       ├── Fallback
    │       │   ├── HighHPCombatLogic (SubTree) — 高血量进攻
    │       │   └── MediumHPCombatLogic (SubTree) — 中血量策略
    │       └── ReactiveFallback (非战斗时占点)
    │           ├── ReactiveSequence (低血量→补给区回血)
    │           │   ├── IsHPBelow: HP < 200?
    │           │   ├── RateController(1Hz) → SendGoal(补给区)
    │           │   └── RobotControl(云台扫描, 底盘旋转)
    │           └── ReactiveSequence (抢控制区)
    │               ├── RateController(1Hz) → SendGoal(控制区)
    │               └── RobotControl(云台扫描, 底盘旋转)
    └── [不满足] ReactiveSequence (非比赛→回家)
        ├── RateController(1Hz) → SendGoal(Home: 0,0)
        └── RobotControl(停止扫描, 停止旋转)
```

---

**文档生成时间**: 2026-02-12  
**数据来源**: `rm_behavior_tree/rm_behavior_tree/config/3v3_new.xml` + 对应源码

## 11. 📝 修复记录

以下为本次代码审查后自动修复的所有文件变更记录：

### HPP 文件修改 (4个)

| 文件路径 | Bug# | 修改内容 |
|----------|------|----------|
| `include/rm_behavior_tree/plugins/action/sub_game_status.hpp` | #6 | include guard: `SUB_ALL_ROBOT_HP_HPP_` → `SUB_GAME_STATUS_HPP_`（#ifndef, #define, #endif 三处） |
| `include/rm_behavior_tree/plugins/action/keep_running.hpp` | #2 | 基类: `SyncActionNode` → `StatefulActionNode`；方法声明: `tick()` → `onStart()/onRunning()/onHalted()` |
| `include/rm_behavior_tree/plugins/action/move_around.hpp` | #3 | 添加成员变量: `std::chrono::time_point<std::chrono::high_resolution_clock> last_goal_time_;` |
| `include/rm_behavior_tree/plugins/condition/is_attacked.hpp` | #1 | 类名: `IsAttakedAction` → `IsAttackedAction`；构造函数声明同步更名 |

### CPP 文件修改 (7个)

| 文件路径 | Bug# | 修改内容 |
|----------|------|----------|
| `plugins/action/keep_running.cpp` | #2 | 构造函数基类: `SyncActionNode` → `StatefulActionNode`；`tick()` 替换为 `onStart()` 返回 RUNNING + `onRunning()` 返回 RUNNING + `onHalted()` 空实现 |
| `plugins/action/move_around.cpp` | #3 | `onStart()`: 添加 `last_goal_time_` 初始化（当前时间-1000ms）；`onRunning()`: 移除 `sleep_for(1000ms)`，添加 `elapsed < 1000` 判断返回 RUNNING，发送目标后更新 `last_goal_time_` |
| `plugins/action/send_goal.cpp` | #12 | `msg.header.frame_id`: `"chassis"` → `"map"` |
| `plugins/action/sub_robot_position.cpp` | #9 | 移除 `robot_position_callback()` 中整个 `bb->set("pose.x/y/yaw")` 代码块（含 try-catch 约14行） |
| `plugins/condition/is_attacked.cpp` | #1 | 类名全替换: `IsAttakedAction` → `IsAttackedAction`；工厂注册: `"IsAttaked"` → `"IsAttacked"` |
| `plugins/decorator/rate_controller.cpp` | #10 | `tick()` 开头添加: `double hz=1.0; getInput("hz",hz); if(hz>0.0) { period_=1.0/hz; }` |
| `plugins/control/decision_switch.cpp` | #11 | 添加 `const size_t num_children = children_nodes_.size();`；case 1/2 各添加 `if(num_children < N)` 越界检查 |

### XML 文件修改 (17个)

| 文件路径 | Bug# | 修改内容 |
|----------|------|----------|
| `config/3v3_new.xml` | #1,8,15,16 | ① 注释掉12个无源文件节点声明；② `IsAttaked`→`IsAttacked`；③ `NotArrived` 端口 `arrived`→`rfid_status`；④ `IsStatusOK` 端口重写对齐C++ |
| `config/7v7.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/attack_left.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/decision_switch_test.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/HighHP_combat_logic.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/IsDetectEnemy.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/protect_supply.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/retreat_attack_left.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `config/tesk.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `rmuc_01.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `previouse_config/7.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `previouse_config/attack_left.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `previouse_config/protect_supply.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `previouse_config/retreat_attack_left.xml` | #1 | `IsAttaked` → `IsAttacked` |
| `previouse_config/tesk.xml` | #1 | `IsAttaked` → `IsAttacked` |

### 修改统计
- **HPP**: 4 文件
- **CPP**: 7 文件  
- **XML**: 17 文件（15 个仅改拼写，2 个有端口/节点声明修改）
- **总计**: 28 个文件
- **修复 Bug 数**: 11/16（剩余 5 个为设计/配置层面问题）

---